# S-Fig 2 — Saturation Curves (all heads overlaid)

AUROC vs context length, LSTM / Transformer / MeanPool on the same axes.  
**Source**: analysis.csv (k=all)  
**Tasks**: main tasks

In [ ]:
import sys
from pathlib import Path

# ── Workspace root (parent of NSRR-tools/) ────────────────────────────────────
WORKSPACE_ROOT = Path("../../../../..").resolve()   # adjust if notebook depth differs
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path
sys.path.insert(0, str(Path(".").resolve()))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib
matplotlib.use("Agg")   # comment out in Jupyter to get inline plots
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()
print("Setup OK — workspace root:", WORKSPACE_ROOT)

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
TASKS  = MAIN_TASKS
HEADS  = ["lstm", "transformer", "mean_pool"]
SPLIT  = "test"
N_COLS = 3
N_ROWS = (len(TASKS) + N_COLS - 1) // N_COLS

df = load_analysis("phase0_v3", split=SPLIT, k="all")

In [ ]:
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(FULL_W, N_ROWS * 2.1))
axes_flat = axes.flatten()

for i, (ax, task) in enumerate(zip(axes_flat, TASKS)):
    panels.saturation_panel(ax, df, task, heads=HEADS, show_values=True)
    ax.set_title(TASK_LABEL[task], fontsize=8)
    add_panel_label(ax, f"({chr(97+i)})")
    # Only first panel gets legend
    if i > 0 and ax.get_legend():
        ax.get_legend().remove()

# Shared legend
handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=7,
           bbox_to_anchor=(0.5, 1.02), frameon=False)

for ax in axes_flat[len(TASKS):]:
    ax.set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.98], h_pad=1.5, w_pad=1.0)
save_figure(fig, FINAL_OUT, "sfig2_saturation")
plt.show()